# Phase 3: Hierarchical Segmentation (Line → Word → Character)
### Simple, Step-by-Step Interactive Guide & Visual Walkthrough
---
### What is Phase 3 and why do we need it?
A neural network classifier cannot read an entire handwritten page all at once. It can only classify **one individual letter or digit at a time** (like in MNIST or EMNIST).

Therefore, before we can classify handwriting, we must slice the document into smaller pieces in a clean, logical hierarchy:
1. **Lines:** Separate the page into individual lines of text (from top to bottom).
2. **Words:** Group characters in each line into distinct words separated by spaces (from left to right).
3. **Characters:** Extract individual character bounding boxes (letters, digits, punctuation).

**Phase 3 achieves this through 4 classical algorithms:**
- **Horizontal Projection Profile (HPP):** Summing ink pixels across rows to find valleys between lines.
- **Morphological Bridging:** Dilating horizontally to merge letters into words while keeping inter-word spaces.
- **Connected Component Analysis (CCA):** Finding isolated ink strokes within each word.
- **Noise & Diacritic Handling:** Merging dots on 'i' and 'j' with their stems, and filtering microscopic dust speckles.

In [ ]:
# Step 1: Import required libraries and project modules
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model directory to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import ingest_image
from preprocessing import preprocess_image
from segmentation import (
    compute_horizontal_projection,
    detect_lines,
    detect_words,
    extract_characters,
    segment_document,
    render_segmentation_overlay,
    SegmentationEngine,
)
import config

print("All Phase 3 segmentation modules loaded successfully!")

---
## Step 2: Preparing Clean Binary Handwriting (Phase 1 & Phase 2)
We ingest our test photograph and preprocess it to produce a clean inverted binary mask ($255 = \text{ink}$, $0 = \text{paper}$).

In [ ]:
# Load test photograph
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Phase 1: Ingestion
orig_img, ingested_img, meta = ingest_image(sample_file)

# Phase 2: Illumination-robust preprocessing
prep_result = preprocess_image(ingested_img)
binary_mask = prep_result.binary_mask
gray_img = prep_result.gray_image

print(f"Binary Mask Shape : {binary_mask.shape} (Height, Width)")
print(f"Total Ink Pixels  : {(binary_mask == 255).sum():,}")

---
## Step 3: Detecting Lines using Horizontal Projection Profiles (HPP)
How does the computer find where a line starts and ends?

We calculate the **Horizontal Projection Profile (HPP)**: for every row $y$, we count how many white ink pixels exist across all columns.
- **Peaks:** Rows that pass through text lines contain lots of ink pixels.
- **Valleys:** Blank white spaces between lines have almost zero ink pixels.

In [ ]:
# Calculate row-wise ink sum
hpp = compute_horizontal_projection(binary_mask)

# Detect text line boundaries
line_boxes = detect_lines(binary_mask)
print(f"Detected {len(line_boxes)} lines of handwriting:")
for i, (lx, ly, lw, lh) in enumerate(line_boxes):
    print(f"  Line {i+1}: y = {ly} to {ly+lh} (Height = {lh} px, Width = {lw} px)")

# Plot the HPP curve
plt.figure(figsize=(10, 4))
plt.plot(hpp, color="navy", linewidth=1.5)
plt.title("Horizontal Projection Profile (Ink Density per Row)", fontsize=12, fontweight="bold")
plt.xlabel("Row Index (y)")
plt.ylabel("Ink Pixel Count")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## Step 4: Grouping Characters into Words (Morphological Bridging)
Letters inside the same word have small gaps between them. But different words have larger spaces between them.

To separate words automatically:
1. We apply a small horizontal dilation with a rectangular kernel `cv2.dilate(line, (15, 1))`.
2. This **bridges adjacent letters** belonging to the same word into a single continuous block.
3. The natural spaces between words are wider, so they do **not** bridge together!

In [ ]:
# Take Line 1 as an example
first_line_bbox = line_boxes[0]
lx, ly, lw, lh = first_line_bbox
line_slice = binary_mask[ly:ly+lh, lx:lx+lw]

# Detect words in this line
word_boxes = detect_words(line_slice, line_offset=(lx, ly))
print(f"Detected {len(word_boxes)} words in Line 1:")
for idx, (wx, wy, ww, wh) in enumerate(word_boxes):
    print(f"  Word {idx+1}: x = {wx}, y = {wy}, w = {ww}, h = {wh}")

---
## Step 5: Extracting Individual Character Boxes & Cleaning Noise
Inside each word, we extract connected ink components (`cv2.connectedComponentsWithStats`).

Our pipeline automatically applies:
1. **Speckle Rejection:** Removes tiny dust specks ($< 15\text{px}^2$).
2. **Ruling Line Rejection:** Discards scratches and long horizontal lines ($w/h > 8.0$).
3. **Diacritic Merging:** Unifies the floating dot on an 'i' or 'j' with its lower stem into one box.
4. **Ligature Splitting:** If two cursive characters are joined, it detects the thin connecting valley and splits them.

In [ ]:
# Run full hierarchical segmentation on the entire page
seg_engine = SegmentationEngine()
seg_result = seg_engine.segment(binary_mask, gray_image=gray_img)

print(f"Total Lines Detected      : {seg_result.total_lines}")
print(f"Total Words Detected      : {seg_result.total_words}")
print(f"Total Characters Extracted : {seg_result.total_characters}")

---
## Step 6: Diagnostic Visual Overlay (Lines, Words, and Characters)
Let us render our multi-level segmentation overlay:
- **Blue Boxes:** Text Lines
- **Green Boxes:** Words
- **Red Boxes:** Individual Characters

In [ ]:
# Render diagnostic overlay on the straightened document
overlay = render_segmentation_overlay(ingested_img, seg_result)

plt.figure(figsize=(15, 9))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.title("Hierarchical Segmentation: Lines (Blue) | Words (Green) | Characters (Red)", fontsize=13, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

---
## Step 7: Inspecting Extracted Character Crops
Let us display a sample grid of isolated character crops ready for Phase 4 normalization.

In [ ]:
all_chars = seg_result.get_all_characters()
sample_chars = all_chars[:16] # First 16 characters

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for idx, char_crop in enumerate(sample_chars):
    ax = axes[idx // 8, idx % 8]
    ax.imshow(char_crop.binary_crop, cmap="gray")
    ax.set_title(f"Char #{idx+1}\n({char_crop.width}x{char_crop.height})", fontsize=9)
    ax.axis("off")

plt.suptitle("Sample Extracted Character Crops (Binary Ink Masks)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Quick Professor & Viva Cheat Sheet (Phase 3)

> **Q1: What is the main purpose of Phase 3?**  
> *Answer:* To recursively segment the binary document into a structured hierarchy of Lines $\to$ Words $\to$ Characters in reading order, so individual character crops can be normalized and classified.
>
> **Q2: How does Horizontal Projection Profile (HPP) detect lines?**  
> *Answer:* It sums foreground ink pixels horizontally across each row. Rows with high values correspond to text lines, while valleys with near-zero values indicate blank spaces separating lines.
>
> **Q3: How are words separated from individual letters?**  
> *Answer:* We apply a horizontal morphological dilation with a rectangular structuring element. This bridges small character spaces within a word together, while leaving larger spaces between words open.
>
> **Q4: How do you prevent dirt and dust specks from being classified as characters?**  
> *Answer:* We use a minimum pixel area threshold (`MIN_CHAR_AREA = 15 px^2`) and minimum dimension thresholds (`width >= 3 px`, `height >= 5 px`) to discard noise components.
>
> **Q5: What happens to dots on 'i' and 'j'?**  
> *Answer:* The algorithm checks for vertical alignment and merges diacritic components located directly above a main character stem into one unified character bounding box.